# HumAID — Zero-shot Classification (Filtered Labels, Batch API, Sharding)

- **Filtered labels (per event):** prompts + JSON schema only list labels that appear in that event’s ground truth → reduces out-of-scope (OOS) predictions.
- **Batch API flow:** build `requests.jsonl` → upload → create batch → poll → download `outputs.jsonl` (and `errors.jsonl` if any).
- **Patch pass:** after batch completes, any missing/blank predictions are re-classified synchronously so `predictions.csv` has one row per input.
- **Stratified sharding (optional):** split large events into *k* shards **preserving class ratios**; use the **same** event-level labels + rules for all shards; merge predictions back in original order.
- **Reporting:** confusion matrices (counts + row-normalized), per-class F1/error, mistakes CSV, and a sortable `results/index.html`.  
  - **Scope** = label universe used for metrics (default `truth`).  
  - **OOS preds** = predictions not in the truth set (QA signal).

## Key settings
- `MODEL` (e.g., `gpt-4o`), `RULES` (e.g., `RULES_1`), `TAG`
- `DRYRUN_N`, `POLL_SECS`
- Token budgeting: `BATCH_TOKEN_LIMIT`, `SAFETY_MARGIN`, `MAX_OUTPUT_TOKENS`
- `.env` with `OPENAI_API_KEY_1` (and optionally a second key)

# 0) Setup

In [1]:
from pathlib import Path
import math
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # token budgeting (sampling-based)
from humaidclf import run_experiment_sharded          # NEW: stratified sharded runner
from humaidclf.batch import use_api_key_env           # (optional) keep key switcher
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["test"]             # or ["train","dev","test"]
MODEL = "gpt-4o"
RULES = RULES_1
TAG = "modeS-gpt-4o-RULES1-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

# Token caps & estimates
BATCH_TOKEN_LIMIT = 90_000   # Tier-1 cap
SAFETY_MARGIN = 0.90            # use only 90% of the cap
MAX_OUTPUT_TOKENS = 40          # matches your request schema

# 1) Discover datasets (events/splits)

In [2]:
def discover_tsvs(base: Path, splits: list[str]):
    items = []
    for event_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        event = event_dir.name
        for split in splits:
            tsv = event_dir / f"{event}_{split}.tsv"
            if tsv.exists():
                items.append({"event": event, "split": split, "tsv": str(tsv)})
    return pd.DataFrame(items)

df_sources = discover_tsvs(BASE, SPLITS)

# --- token budgeting ---
token_index = build_token_index(
    df_sources,
    model=MODEL,
    rules_text=RULES,
    batch_token_limit=BATCH_TOKEN_LIMIT,
    safety_margin=SAFETY_MARGIN,
    sample_size=200,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

display(token_index)

df_fit     = token_index[token_index["fits_cap"]].reset_index(drop=True)
df_too_big = token_index[~token_index["fits_cap"]].reset_index(drop=True)

print("OK to run as single batch:")
display(df_fit[["event","split","num_rows","est_total_tokens","limit_used_%"]])

print("Will be sharded (exceeds cap):")
display(df_too_big[["event","split","num_rows","est_total_tokens","limit_used_%"]])

,event,split,tsv,num_rows,avg_req_tokens,est_total_tokens,fits_cap,limit_used_%
1,canada_wildfires_2016,test,Dataset\HumAID\canada_wildfires_2016\canada_wi...,445,473,210485,False,233.9
8,kaikoura_earthquake_2016,test,Dataset\HumAID\kaikoura_earthquake_2016\kaikou...,435,487,211845,False,235.4
2,cyclone_idai_2019,test,Dataset\HumAID\cyclone_idai_2019\cyclone_idai_...,779,519,404301,False,449.2
4,hurricane_florence_2018,test,Dataset\HumAID\hurricane_florence_2018\hurrica...,1241,501,621741,False,690.8
7,hurricane_maria_2017,test,Dataset\HumAID\hurricane_maria_2017\hurricane_...,1442,487,702254,False,780.3
0,california_wildfires_2018,test,Dataset\HumAID\california_wildfires_2018\calif...,1461,507,740727,False,823.0
3,hurricane_dorian_2019,test,Dataset\HumAID\hurricane_dorian_2019\hurricane...,1508,503,758524,False,842.8
9,kerala_floods_2018,test,Dataset\HumAID\kerala_floods_2018\kerala_flood...,1582,508,803656,False,893.0
5,hurricane_harvey_2017,test,Dataset\HumAID\hurricane_harvey_2017\hurricane...,1805,486,877230,False,974.7
6,hurricane_irma_2017,test,Dataset\HumAID\hurricane_irma_2017\hurricane_i...,1862,486,904932,False,1005.5


OK to run as single batch:


,event,split,num_rows,est_total_tokens,limit_used_%


Will be sharded (exceeds cap):


,event,split,num_rows,est_total_tokens,limit_used_%
0,canada_wildfires_2016,test,445,210485,233.9
1,kaikoura_earthquake_2016,test,435,211845,235.4
2,cyclone_idai_2019,test,779,404301,449.2
3,hurricane_florence_2018,test,1241,621741,690.8
4,hurricane_maria_2017,test,1442,702254,780.3
5,california_wildfires_2018,test,1461,740727,823.0
6,hurricane_dorian_2019,test,1508,758524,842.8
7,kerala_floods_2018,test,1582,803656,893.0
8,hurricane_harvey_2017,test,1805,877230,974.7
9,hurricane_irma_2017,test,1862,904932,1005.5


# 2) Run all datasets (sequentially)

In [3]:
def run_list_single(dflist: pd.DataFrame, rules_text: str, model: str, tag: str):
    """Run events that already fit under the cap using the normal runner."""
    results = []
    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        print(f"\n=== Running (single) {event}/{split} ({model} | {tag}) ===")
        try:
            plan, preds, summary = run_experiment(
                dataset_path=tsv,
                rules=rules_text,
                model=model,
                tag=tag,
                dryrun_n=DRYRUN_N,
                poll_secs=POLL_SECS,
                out_root=OUT_ROOT,
                do_analysis=DO_ANALYSIS,
            )
            acc = summary.get("accuracy") if summary else float("nan")
            f1  = summary.get("macro_f1") if summary else float("nan")
            n   = summary.get("num_total_with_truth") if summary else len(preds)
            results.append({
                "event": event, "split": split,
                "run_dir": str(plan["dir"]),
                "predictions_csv": str(plan["predictions_csv"]),
                "macro_f1": f1, "accuracy": acc, "num_total": n,
                "mode": "single",
            })
        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")
            results.append({
                "event": event, "split": split, "run_dir": "ERROR",
                "predictions_csv": "", "macro_f1": float("nan"),
                "accuracy": float("nan"), "num_total": 0, "mode": "single",
            })
    return pd.DataFrame(results)

def run_list_sharded(dflist: pd.DataFrame, token_df: pd.DataFrame, rules_text: str, model: str, tag: str):
    """Run events that exceed the cap using stratified shards. k is computed from token estimates."""
    results = []
    # Build a quick lookup: (event,split) -> est_total_tokens
    est_map = {(r.event, r.split): r.est_total_tokens for r in token_df.itertuples(index=False)}
    eff_cap = BATCH_TOKEN_LIMIT * SAFETY_MARGIN

    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        est_tokens = est_map.get((event, split), None)
        # Conservative shard count: ceil(est / eff_cap). Min 2.
        k = max(2, math.ceil((est_tokens or (eff_cap + 1)) / eff_cap))
        print(f"\n=== Running (sharded x{k}) {event}/{split} ({model} | {tag}) ===")
        try:
            plan, preds, summary = run_experiment_sharded(
                dataset_path=tsv,
                rules=rules_text,
                model=model,
                tag=f"{tag}-sharded{k}",
                k_shards=k,
                temperature=0.0,
                poll_secs=POLL_SECS,
                out_root=OUT_ROOT,
                do_analysis=DO_ANALYSIS,
                analysis_subdir="analysis",  # merged analysis
            )
            acc = summary.get("accuracy") if summary else float("nan")
            f1  = summary.get("macro_f1") if summary else float("nan")
            n   = summary.get("num_total_with_truth") if summary else len(preds)
            results.append({
                "event": event, "split": split,
                "run_dir": str(plan["dir"]),
                "predictions_csv": str(plan["predictions_csv"]),
                "macro_f1": f1, "accuracy": acc, "num_total": n,
                "mode": f"sharded{k}",
            })
        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")
            results.append({
                "event": event, "split": split, "run_dir": "ERROR",
                "predictions_csv": "", "macro_f1": float("nan"),
                "accuracy": float("nan"), "num_total": 0, "mode": f"sharded{k}",
            })
    return pd.DataFrame(results)

In [4]:
# --- Run singles with your normal key (optional context manager kept for parity)
with use_api_key_env("OPENAI_API_KEY_1"):
    print(">>> Using OPENAI_API_KEY_1")
    df_runs_single = run_list_single(df_fit, RULES, MODEL, tag=f"{TAG}-TIER1")

# --- Run sharded for the too-big ones (same key or another if you prefer)
# You can keep the same key; sharding is already controlling token usage.
with use_api_key_env("OPENAI_API_KEY_1"):
    if not df_too_big.empty:
        df_runs_sharded = run_list_sharded(df_too_big, token_index, RULES, MODEL, tag=f"{TAG}")
    else:
        df_runs_sharded = pd.DataFrame()
        print("No large datasets to shard.")

# 3) Save a small index of all runs
from datetime import datetime
idx_dir = Path(OUT_ROOT) / "_indexes"
idx_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")

all_runs = pd.concat([df_runs_single, df_runs_sharded], ignore_index=True)
all_runs.to_csv(idx_dir / f"runs_{MODEL}_{TAG}_{stamp}.csv", index=False)
print("Saved run index at:", idx_dir)
display(all_runs)

>>> Using OPENAI_API_KEY_1

=== Running (sharded x3) canada_wildfires_2016/test (gpt-4o | modeS-gpt-4o-RULES1-filtered) ===
[batch batch_6909c1947804819092b10a6ff648df7d] status = validating
[batch batch_6909c1947804819092b10a6ff648df7d] status = completed
[batch batch_6909c2c326f4819087b1c0abc23dd3b1] status = validating
[batch batch_6909c2c326f4819087b1c0abc23dd3b1] status = completed
[batch batch_6909c3f24ee4819099637d511c79647c] status = validating
[batch batch_6909c3f24ee4819099637d511c79647c] status = completed
Saved merged predictions to: runs\canada_wildfires_2016\test\gpt-4o\20251104-010418-modeS-gpt-4o-RULES1-filtered-sharded3\predictions.csv
Macro-F1 (merged): 0.672588795143568

=== Running (sharded x3) kaikoura_earthquake_2016/test (gpt-4o | modeS-gpt-4o-RULES1-filtered) ===
[batch batch_6909c522d5bc819095390c73ed019e51] status = validating
[batch batch_6909c522d5bc819095390c73ed019e51] status = completed
[batch batch_6909c651ebd08190b6fd413d3d131db2] status = validating
[b

,event,split,run_dir,predictions_csv,macro_f1,accuracy,num_total,mode
0,canada_wildfires_2016,test,runs\canada_wildfires_2016\test\gpt-4o\2025110...,runs\canada_wildfires_2016\test\gpt-4o\2025110...,0.672589,0.786517,445,sharded3
1,kaikoura_earthquake_2016,test,runs\kaikoura_earthquake_2016\test\gpt-4o\2025...,runs\kaikoura_earthquake_2016\test\gpt-4o\2025...,0.726821,0.740230,435,sharded3
2,cyclone_idai_2019,test,runs\cyclone_idai_2019\test\gpt-4o\20251104-01...,runs\cyclone_idai_2019\test\gpt-4o\20251104-01...,0.640507,0.753530,779,sharded5
3,hurricane_florence_2018,test,runs\hurricane_florence_2018\test\gpt-4o\20251...,runs\hurricane_florence_2018\test\gpt-4o\20251...,0.707257,0.768735,1241,sharded8
4,hurricane_maria_2017,test,runs\hurricane_maria_2017\test\gpt-4o\20251104...,runs\hurricane_maria_2017\test\gpt-4o\20251104...,0.634095,0.667129,1442,sharded9
5,california_wildfires_2018,test,runs\california_wildfires_2018\test\gpt-4o\202...,runs\california_wildfires_2018\test\gpt-4o\202...,0.628638,0.712526,1461,sharded10
6,hurricane_dorian_2019,test,runs\hurricane_dorian_2019\test\gpt-4o\2025110...,runs\hurricane_dorian_2019\test\gpt-4o\2025110...,0.598173,0.637268,1508,sharded10
7,kerala_floods_2018,test,ERROR,,NaN,NaN,0,sharded10
8,hurricane_harvey_2017,test,ERROR,,NaN,NaN,0,sharded11
9,hurricane_irma_2017,test,runs\hurricane_irma_2017\test\gpt-4o\20251104-...,runs\hurricane_irma_2017\test\gpt-4o\20251104-...,0.617246,0.634801,1862,sharded12


# Other experiments

In [1]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment_sharded
from humaidclf.batch import use_api_key_env
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
MODEL = "gpt-4o"
RULES = RULES_1
TAG = "modeS-gpt-4o-RULES1-filtered"
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"
K = 10  # number of stratified shards

with use_api_key_env("OPENAI_API_KEY"):
    plan, preds, summary = run_experiment_sharded(
        dataset_path=str(BASE / "kerala_floods_2018" / "kerala_floods_2018_test.tsv"),
        rules=RULES,
        model=MODEL,
        tag=f"{TAG}-sharded{K}",
        k_shards=K,
        temperature=0.0,
        poll_secs=POLL_SECS,
        out_root=OUT_ROOT,
        do_analysis=DO_ANALYSIS,
        analysis_subdir="analysis",
    )

summary



[batch batch_690a38711f548190a5ec1c1d53effe07] status = validating
[batch batch_690a38711f548190a5ec1c1d53effe07] status = completed
[batch batch_690a39a12f088190bdb233e3ab95750f] status = validating
[batch batch_690a39a12f088190bdb233e3ab95750f] status = completed
[batch batch_690a3ad0ff2c8190b6e4e4614786d8e6] status = validating
[batch batch_690a3ad0ff2c8190b6e4e4614786d8e6] status = completed
[batch batch_690a3c0088f481909df28b081f4136bf] status = validating
[batch batch_690a3c0088f481909df28b081f4136bf] status = completed
[batch batch_690a3d2f1a0081909d8e75da87e3fb36] status = validating
[batch batch_690a3d2f1a0081909d8e75da87e3fb36] status = completed
[batch batch_690a3e5dba8881908ba96ad11ce96acf] status = validating
[batch batch_690a3e5dba8881908ba96ad11ce96acf] status = completed
[batch batch_690a3f8c7e248190b3c05e440e0e70a7] status = validating
[batch batch_690a3f8c7e248190b3c05e440e0e70a7] status = completed
[batch batch_690a40bc44dc8190b017194b8052fb2d] status = validating
[b

{'num_total_with_truth': 1582,
 'num_correct': 1093,
 'num_incorrect': 489,
 'accuracy': 0.690897597977244,
 'macro_f1': 0.5599230181223613,
 'labels': ['caution_and_advice',
  'displaced_people_and_evacuations',
  'infrastructure_and_utility_damage',
  'injured_or_dead_people',
  'not_humanitarian',
  'other_relevant_information',
  'requests_or_urgent_needs',
  'rescue_volunteering_or_donation_effort',
  'sympathy_and_support'],
 'labels_scope': 'truth',
 'invalid_pred_outside_truth': 0}

In [1]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment_sharded
from humaidclf.batch import use_api_key_env
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
MODEL = "gpt-4o"
RULES = RULES_1
TAG = "modeS-gpt-4o-RULES1-filtered"
POLL_SECS = 600
DO_ANALYSIS = True
OUT_ROOT = "runs"
K = 10  # number of stratified shards

with use_api_key_env("OPENAI_API_KEY"):
    plan, preds, summary = run_experiment_sharded(
        dataset_path=str(BASE / "hurricane_harvey_2017" / "hurricane_harvey_2017_test.tsv"),
        rules=RULES,
        model=MODEL,
        tag=f"{TAG}-sharded{K}",
        k_shards=K,
        temperature=0.0,
        poll_secs=POLL_SECS,
        out_root=OUT_ROOT,
        do_analysis=DO_ANALYSIS,
        analysis_subdir="analysis",
    )

summary


[batch batch_690a5e9a91d481909d8b2dde43acf56c] status = validating
[batch batch_690a5e9a91d481909d8b2dde43acf56c] status = in_progress
[batch batch_690a5e9a91d481909d8b2dde43acf56c] status = completed
[batch batch_690a635f221081909a8d8ce2af0032e3] status = validating
[batch batch_690a635f221081909a8d8ce2af0032e3] status = completed
[batch batch_690a65bb8d788190a3c8b62b58bb2cda] status = validating
[batch batch_690a65bb8d788190a3c8b62b58bb2cda] status = in_progress
[batch batch_690a65bb8d788190a3c8b62b58bb2cda] status = completed
[batch batch_690a6a7125cc8190a59bea7aa18cebce] status = validating
[batch batch_690a6a7125cc8190a59bea7aa18cebce] status = completed
[batch batch_690a6ccc5c648190833c2ad271df78ef] status = validating
[batch batch_690a6ccc5c648190833c2ad271df78ef] status = in_progress
[batch batch_690a6ccc5c648190833c2ad271df78ef] status = completed
[batch batch_690a71876ff48190b24c3b234d45ce4d] status = validating
[batch batch_690a71876ff48190b24c3b234d45ce4d] status = complete

{'num_total_with_truth': 1805,
 'num_correct': 1187,
 'num_incorrect': 618,
 'accuracy': 0.657617728531856,
 'macro_f1': 0.6129852649572137,
 'labels': ['caution_and_advice',
  'displaced_people_and_evacuations',
  'infrastructure_and_utility_damage',
  'injured_or_dead_people',
  'not_humanitarian',
  'other_relevant_information',
  'requests_or_urgent_needs',
  'rescue_volunteering_or_donation_effort',
  'sympathy_and_support'],
 'labels_scope': 'truth',
 'invalid_pred_outside_truth': 0}